#  Pac-Man

In [ ]:
!pip -q install "gymnasium[atari]>=1.0.0" tensorboard moviepy

## 1. Connect Google Drive 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Settings and the game


In [ ]:
import os, time, csv, random, base64, glob
from collections import deque
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym
import ale_py
from gymnasium.wrappers import AtariPreprocessing, FrameStackObservation
from torch.utils.tensorboard import SummaryWriter

gym.register_envs(ale_py)
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = True
print("device:", device)
if device == "cpu":
    print("WARNING: no GPU. Use Runtime > Change runtime type > T4 GPU, or training will be very slow.")


@dataclass
class Config:
    run_dir: str = "/content/drive/MyDrive/pacman_dqn/run2_bigcnn"   # everything is saved here
    env_id: str = "ALE/MsPacman-v5"
    seed: int = 42

    # training length
    total_steps: int = 6_000_000       # agent decisions (each = 4 game frames)
    warmup_steps: int = 20_000         # random-ish play before learning starts
    learn_every: int = 4
    target_update_every: int = 10_000

    # exploration (random-move chance)
    eps_start: float = 1.0
    eps_end: float = 0.02
    eps_decay_steps: int = 1_000_000

    # learning
    lr: float = 1e-4
    lr_final: float = 1e-5             # learning rate at the very end
    lr_decay_start: int = 3_500_000    # stays at lr until here, then falls linearly to lr_final
    gamma: float = 0.99
    n_step: int = 3                    # look 3 steps ahead when computing targets
    batch_size: int = 32
    buffer_size: int = 500_000         # frames stored once each: about 3.5 GB RAM

    # network: sizes of the hidden layers in each head. (512,) is standard; try (512, 512)
    hidden_sizes: tuple = (512,)
    conv_channels: tuple = (64, 128, 128)   # (32, 64, 64) = the small network from run1
    extra_conv: int = 2                     # extra 3x3 conv layers (0 = small network)

    # evaluation, logging, saving
    eval_every: int = 250_000
    eval_games: int = 20
    eval_max_steps: int = 10_000       # stops an evaluation game where the AI gets stuck
    log_every: int = 10_000
    checkpoint_every: int = 100_000
    resume: bool = True                # continue from run_dir/latest.pt if it exists


def make_env(cfg, render_mode=None):
    env = gym.make(cfg.env_id, frameskip=1, render_mode=render_mode)
    env = AtariPreprocessing(env, noop_max=30, frame_skip=4, screen_size=84, grayscale_obs=True)
    env = FrameStackObservation(env, stack_size=4)       # observation: (4, 84, 84) uint8
    return env

## 3. The AI's memory


In [ ]:
class ReplayBuffer:
    """Stores every frame ONCE (about 7 KB per step instead of 56 KB), so a much bigger memory fits in RAM.
    The 4-frame stacks and the n-step returns are rebuilt when a batch is sampled."""

    def __init__(self, capacity, n_step, gamma, stack=4):
        self.capacity, self.n, self.gamma, self.stack = capacity, n_step, gamma, stack
        self.frames  = np.zeros((capacity, 84, 84), dtype=np.uint8)  # newest frame of the state at each step
        self.actions = np.zeros(capacity, dtype=np.int64)
        self.rewards = np.zeros(capacity, dtype=np.float32)          # clipped reward after the action
        self.ended   = np.zeros(capacity, dtype=bool)                # life lost / game over after this step
        self.first   = np.zeros(capacity, dtype=bool)                # this state is the first of an episode
        self.pos = 0
        self.size = 0

    def add(self, frame, action, reward, ended, first):
        i = self.pos
        self.frames[i], self.actions[i], self.rewards[i] = frame, action, reward
        self.ended[i], self.first[i] = ended, first
        self.pos = (self.pos + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def __len__(self):
        return self.size

    def _stack_indices(self, t):
        """Indices of the 4 frames of the state at t (oldest to newest). At an episode start the
        first frame is repeated, exactly like the environment's frame stack does."""
        idxs, cur = [t], t
        for _ in range(self.stack - 1):
            cur = np.where(self.first[cur], cur, (cur - 1) % self.capacity)
            idxs.append(cur)
        return np.stack(idxs[::-1], axis=1)

    def _pick(self, batch):
        if self.size < self.stack + self.n + 1:
            raise RuntimeError("replay buffer too small to sample from yet")
        oldest = (self.pos - self.size) % self.capacity
        offsets = np.random.randint(self.stack - 1, self.size - self.n, size=batch)
        return (oldest + offsets) % self.capacity

    def _gather(self, t):
        total = np.zeros(len(t), dtype=np.float32)
        alive = np.ones(len(t), dtype=bool)
        g = 1.0
        for k in range(self.n):                      # discounted reward over the next n steps,
            j = (t + k) % self.capacity              # stopping at a life loss / game over
            total += alive * g * self.rewards[j]
            alive &= ~self.ended[j]
            g *= self.gamma
        bootstrap = (alive * g).astype(np.float32)   # 0 if the episode ended, else gamma^n
        states = self.frames[self._stack_indices(t)]
        next_states = self.frames[self._stack_indices((t + self.n) % self.capacity)]
        return states, self.actions[t], total, next_states, bootstrap

    def sample(self, batch):
        return self._gather(self._pick(batch))

## 4. The brain and the agent


In [ ]:
def mlp(in_size, hidden_sizes, out_size):
    layers, size = [], in_size
    for h in hidden_sizes:
        layers += [nn.Linear(size, h), nn.ReLU()]
        size = h
    layers.append(nn.Linear(size, out_size))
    return nn.Sequential(*layers)


class QNet(nn.Module):
    def __init__(self, n_actions, hidden_sizes=(512,), conv_channels=(32, 64, 64), extra_conv=0):
        super().__init__()
        c1, c2, c3 = conv_channels
        layers = [
            nn.Conv2d(4, c1, kernel_size=8, stride=4), nn.ReLU(),     # 84x84 -> 20x20
            nn.Conv2d(c1, c2, kernel_size=4, stride=2), nn.ReLU(),    # 20x20 -> 9x9
            nn.Conv2d(c2, c3, kernel_size=3, stride=1), nn.ReLU(),    # 9x9   -> 7x7
        ]
        for _ in range(extra_conv):                                   # extra layers keep the 7x7 size
            layers += [nn.Conv2d(c3, c3, kernel_size=3, padding=1), nn.ReLU()]
        layers.append(nn.Flatten())
        self.conv = nn.Sequential(*layers)

        flat = c3 * 7 * 7
        self.value = mlp(flat, hidden_sizes, 1)               # how good is this situation?
        self.advantage = mlp(flat, hidden_sizes, n_actions)   # how much better is each move?

    def forward(self, x):
        z = self.conv(x.float() / 255.0)
        v, a = self.value(z), self.advantage(z)
        return v + a - a.mean(dim=1, keepdim=True)


class Agent:
    def __init__(self, n_actions, obs_shape, cfg):
        self.cfg, self.n_actions = cfg, n_actions
        self.q = QNet(n_actions, cfg.hidden_sizes, cfg.conv_channels, cfg.extra_conv).to(device)
        self.target = QNet(n_actions, cfg.hidden_sizes, cfg.conv_channels, cfg.extra_conv).to(device)
        self.target.load_state_dict(self.q.state_dict())
        self.target.eval()
        self.opt = torch.optim.Adam(self.q.parameters(), lr=cfg.lr, eps=1.5e-4)
        self.buffer = ReplayBuffer(cfg.buffer_size, cfg.n_step, cfg.gamma)

    def epsilon(self, step):
        frac = min(1.0, step / self.cfg.eps_decay_steps)
        return self.cfg.eps_start + frac * (self.cfg.eps_end - self.cfg.eps_start)

    def lr_at(self, step):
        c = self.cfg
        if step <= c.lr_decay_start:
            return c.lr
        frac = min(1.0, (step - c.lr_decay_start) / max(1, c.total_steps - c.lr_decay_start))
        return c.lr + frac * (c.lr_final - c.lr)

    @torch.no_grad()
    def act(self, state, eps):
        if random.random() < eps:
            return random.randrange(self.n_actions)
        s = torch.as_tensor(np.asarray(state), device=device).unsqueeze(0)
        return self.q(s).argmax(dim=1).item()

    def learn(self):
        s, a, r, s2, boot = self.buffer.sample(self.cfg.batch_size)
        s    = torch.as_tensor(s,    device=device)
        s2   = torch.as_tensor(s2,   device=device)
        a    = torch.as_tensor(a,    device=device)
        r    = torch.as_tensor(r,    device=device)
        boot = torch.as_tensor(boot, device=device)

        predicted = self.q(s).gather(1, a.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            best_next = self.q(s2).argmax(dim=1, keepdim=True)                # main net picks the move
            next_value = self.target(s2).gather(1, best_next).squeeze(1)      # target net scores it
            wanted = r + boot * next_value                                    # boot = gamma^n, or 0 at episode end

        loss = F.smooth_l1_loss(predicted, wanted)
        self.opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.q.parameters(), 10.0)
        self.opt.step()
        return loss.item()

    def sync_target(self):
        self.target.load_state_dict(self.q.state_dict())

## 5. Evaluation, checkpoints and the training loop


In [ ]:
@torch.no_grad()
def evaluate(net, cfg, n_games, seed=10_000, eps=0.01):
    env = make_env(cfg)
    net.eval()
    scores = []
    for i in range(n_games):
        rng = random.Random(seed + i)
        state, _ = env.reset(seed=seed + i)
        score, steps, done = 0.0, 0, False
        while not done and steps < cfg.eval_max_steps:
            if rng.random() < eps:
                action = rng.randrange(env.action_space.n)
            else:
                s = torch.as_tensor(np.asarray(state), device=device).unsqueeze(0)
                action = net(s).argmax(dim=1).item()
            state, reward, terminated, truncated, _ = env.step(action)
            score += reward
            steps += 1
            done = terminated or truncated
        scores.append(score)
    env.close()
    scores = np.array(scores)
    se = scores.std(ddof=1) / np.sqrt(len(scores)) if len(scores) > 1 else 0.0
    return float(scores.mean()), float(se)


def save_ckpt(path, agent, step, best_eval, cfg):
    tmp = path + ".tmp"
    torch.save({"q": agent.q.state_dict(), "target": agent.target.state_dict(),
                "opt": agent.opt.state_dict(), "step": step, "best_eval": best_eval,
                "hidden_sizes": tuple(cfg.hidden_sizes),
                "conv_channels": tuple(cfg.conv_channels), "extra_conv": cfg.extra_conv}, tmp)
    os.replace(tmp, path)            # atomic: never leaves a half-written file


def load_net(path, cfg):
    ck = torch.load(path, map_location=device, weights_only=False)
    env = make_env(cfg); n_actions = env.action_space.n; env.close()
    net = QNet(n_actions, ck["hidden_sizes"], ck.get("conv_channels", (32, 64, 64)), ck.get("extra_conv", 0)).to(device)
    net.load_state_dict(ck["q"])
    net.eval()
    return net


def train(cfg):
    os.makedirs(cfg.run_dir, exist_ok=True)
    random.seed(cfg.seed); np.random.seed(cfg.seed); torch.manual_seed(cfg.seed)

    env = make_env(cfg)
    agent = Agent(env.action_space.n, env.observation_space.shape, cfg)
    latest_path = os.path.join(cfg.run_dir, "latest.pt")
    best_path = os.path.join(cfg.run_dir, "best.pt")
    start_step, best_eval = 0, -float("inf")

    if cfg.resume and os.path.exists(latest_path):
        ck = torch.load(latest_path, map_location=device, weights_only=False)
        same_layout = (tuple(ck["hidden_sizes"]) == tuple(cfg.hidden_sizes)
                       and tuple(ck.get("conv_channels", (32, 64, 64))) == tuple(cfg.conv_channels)
                       and ck.get("extra_conv", 0) == cfg.extra_conv)
        if not same_layout:
            raise ValueError("latest.pt was trained with a different network layout than Config. "
                             "Use a new run_dir, or set resume=False.")
        agent.q.load_state_dict(ck["q"]); agent.target.load_state_dict(ck["target"])
        agent.opt.load_state_dict(ck["opt"])
        start_step, best_eval = ck["step"], ck["best_eval"]
        print(f"Resuming from step {start_step:,} (best eval so far {best_eval:.0f})")
    else:
        print("Starting a new run in", cfg.run_dir)

    writer = SummaryWriter(os.path.join(cfg.run_dir, "tb"))
    log_path = os.path.join(cfg.run_dir, "scores.csv")
    new_log = not os.path.exists(log_path)
    log_file = open(log_path, "a", newline="")
    log = csv.writer(log_file)
    if new_log:
        log.writerow(["step", "score"])

    state, info = env.reset(seed=cfg.seed + start_step)
    lives = info.get("lives", 3)
    first = True                          # is `state` the first frame of an episode?
    score = 0.0
    recent = deque(maxlen=20)
    learn_start = start_step + cfg.warmup_steps
    last_loss, learn_calls, lr, t0 = 0.0, 0, cfg.lr, time.time()
    step = start_step

    try:
        for step in range(start_step + 1, cfg.total_steps + 1):
            eps = agent.epsilon(step)
            action = agent.act(state, eps)
            next_state, reward, terminated, truncated, info = env.step(action)
            score += reward

            new_lives = info.get("lives", lives)
            life_lost = new_lives < lives
            lives = new_lives

            # Learning sees a life loss as an episode end; the game itself carries on.
            ended = terminated or truncated or life_lost
            agent.buffer.add(state[-1], action, float(np.sign(reward)), ended, first)
            first = False
            state = next_state

            if terminated or truncated:
                recent.append(score)
                log.writerow([step, score]); log_file.flush()
                writer.add_scalar("game/score", score, step)
                writer.add_scalar("game/avg_score_20", float(np.mean(recent)), step)
                score = 0.0
                state, info = env.reset()
                lives = info.get("lives", 3)
                first = True

            if step > learn_start and step % cfg.learn_every == 0:
                lr = agent.lr_at(step)                    # learning-rate decay
                for group in agent.opt.param_groups:
                    group["lr"] = lr
                last_loss = agent.learn()
                learn_calls += 1
                if learn_calls % 500 == 0:
                    writer.add_scalar("train/loss", last_loss, step)

            if step % cfg.target_update_every == 0:
                agent.sync_target()

            if step % cfg.log_every == 0:
                avg = float(np.mean(recent)) if recent else 0.0
                sps = (step - start_step) / (time.time() - t0)
                print(f"step {step:>10,} | avg score (last 20) {avg:7.1f} | eps {eps:.2f} | lr {lr:.1e} | "
                      f"loss {last_loss:.4f} | {sps:,.0f} steps/s")
                writer.add_scalar("train/epsilon", eps, step)
                writer.add_scalar("train/lr", lr, step)
                writer.flush()

            if step % cfg.eval_every == 0:
                mean, se = evaluate(agent.q, cfg, cfg.eval_games)
                agent.q.train()
                writer.add_scalar("eval/score", mean, step)
                print(f"  EVAL @ {step:,}: {mean:.0f} ± {se:.0f}  (best so far {best_eval:.0f})")
                if mean > best_eval:
                    best_eval = mean
                    save_ckpt(best_path, agent, step, best_eval, cfg)
                    print("  -> new best, saved best.pt")

            if step % cfg.checkpoint_every == 0:
                save_ckpt(latest_path, agent, step, best_eval, cfg)

    except KeyboardInterrupt:
        print("Interrupted: saving progress...")
    finally:
        save_ckpt(latest_path, agent, step, best_eval, cfg)
        log_file.close(); writer.close(); env.close()
    print(f"Stopped at step {step:,}. Run the training cell again to continue.")

## 6. Quick smoke test 



In [ ]:
test_cfg = Config(run_dir="/content/smoke_test", total_steps=12_000, warmup_steps=1_000,
                  eps_decay_steps=6_000, eval_every=6_000, eval_games=2, log_every=2_000,
                  checkpoint_every=6_000, buffer_size=5_000, resume=False)
train(test_cfg)

## 7. Live TensorBoard

Run this **before** the training cell. The graphs appear below and refresh while training runs.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/pacman_dqn

## 8. Train


In [1]:
cfg = Config()
train(cfg)

NameError: name 'Config' is not defined

## 9. Learning curve

In [ ]:
cfg = globals().get("cfg") or Config()     # works even after a reconnect
import matplotlib.pyplot as plt

steps, scores = [], []
with open(os.path.join(cfg.run_dir, "scores.csv")) as f:
    for row in csv.DictReader(f):
        steps.append(int(row["step"])); scores.append(float(row["score"]))
scores = np.array(scores)
window = 20

plt.figure(figsize=(9, 5))
plt.plot(steps, scores, alpha=0.3, label="each game (with exploration moves)")
if len(scores) >= window:
    avg = np.convolve(scores, np.ones(window) / window, mode="valid")
    plt.plot(steps[window - 1:], avg, linewidth=2, label=f"{window}-game average")
plt.xlabel("training steps"); plt.ylabel("game score"); plt.legend(); plt.grid(alpha=0.3)
plt.title("Ms. Pac-Man DQN learning curve")
plt.show()

## 10. Final test: best vs. latest 
If the two averages differ by less than about twice their ± value, treat them as tied.

In [ ]:
cfg = globals().get("cfg") or Config()
for name in ["best.pt", "latest.pt"]:
    net = load_net(os.path.join(cfg.run_dir, name), cfg)
    mean, se = evaluate(net, cfg, n_games=100)
    print(f"{name:10s}: {mean:7.1f} ± {se:.1f}  (100 games)")

## 11. Watch 

In [ ]:
cfg = globals().get("cfg") or Config()
from gymnasium.wrappers import RecordVideo
from IPython.display import HTML, display

def record_game(net, cfg, seed=123, folder="/content/videos", max_steps=1500):
    env = make_env(cfg, render_mode="rgb_array")
    env = RecordVideo(env, folder, episode_trigger=lambda e: True,
                      name_prefix=f"seed{seed}", fps=15)      # 15 decisions/s = real game speed
    rng = random.Random(seed)
    state, _ = env.reset(seed=seed)
    done, steps, score = False, 0, 0.0
    with torch.no_grad():
        while not done and steps < max_steps:
            if rng.random() < 0.01:
                action = rng.randrange(env.action_space.n)
            else:
                s = torch.as_tensor(np.asarray(state), device=device).unsqueeze(0)
                action = net(s).argmax(dim=1).item()
            state, r, term, trunc, _ = env.step(action)
            score += r; steps += 1; done = term or trunc
    env.close()
    print(f"score in this clip: {score:.0f}")
    return sorted(glob.glob(f"{folder}/seed{seed}*.mp4"))[-1]

net = load_net(os.path.join(cfg.run_dir, "best.pt"), cfg)
path = record_game(net, cfg, seed=123)
data = base64.b64encode(open(path, "rb").read()).decode()
display(HTML(f'<video width="420" controls autoplay loop><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'))